# Data Integration

This notebook:

- validates processed datasets,
- verifies consistency of merge keys,
- integrates CO₂ emissions and electricity price data,
- creates the final analytical dataset for 2007–2024.

## Importing the processed files

In [1]:
import pandas as pd

df_co2 = pd.read_csv("../data/processed/co2_poland_eu.csv")
df_electricity = pd.read_csv("../data/processed/electricity_prices_poland_eu.csv")

## Performing checks

### Checking columns names

In [2]:
df_co2.columns

Index(['country', 'year', 'population', 'co2', 'co2_per_capita'], dtype='object')

In [3]:
df_electricity.columns

Index(['country', 'year', 'electricity_price'], dtype='object')

### Checking data types

In [4]:
df_co2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         70 non-null     object 
 1   year            70 non-null     int64  
 2   population      70 non-null     float64
 3   co2             70 non-null     float64
 4   co2_per_capita  70 non-null     float64
dtypes: float64(3), int64(1), object(1)
memory usage: 2.9+ KB


In [5]:
df_electricity.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country            36 non-null     object 
 1   year               36 non-null     int64  
 2   electricity_price  36 non-null     float64
dtypes: float64(1), int64(1), object(1)
memory usage: 992.0+ bytes


#### Verifying population column

In [6]:
df_co2["population"].isna().sum()

0

In [7]:
(df_co2["population"] % 1 > 0).any()

False

## Checking unique values under country column

In [8]:
df_electricity["country"].unique()

array(['European Union (27)', 'Poland'], dtype=object)

In [9]:
df_co2["country"].unique()

array(['European Union (27)', 'Poland'], dtype=object)

### Schema inconsistency identified

During the data quality checks, an inconsistency was identified in the `country` values between the processed datasets (`PL` vs `Poland`, `EU27_2020` vs `European Union (27)`).

The issue will be resolved in the transformation notebook to ensure standardized processed datasets before data integration.

### Country issue resolved. Checking year ranges between the files.

In [10]:
df_electricity_year_check_min = df_electricity["year"].min()
df_electricity_year_check_max = df_electricity["year"].max()
df_co2_year_check_min = df_co2["year"].min()
df_co2_year_check_max = df_co2["year"].max()

print(f"electricity df starts on {df_electricity_year_check_min},"
     f" co2 df starts on {df_co2_year_check_min}")

print(f"electricity df end on year {df_electricity_year_check_max},"
     f" co2 df end on {df_co2_year_check_max}")
    

electricity df starts on 2007, co2 df starts on 1990
electricity df end on year 2024, co2 df end on 2024


### Historical CO₂ data is available from 1990, while comparable electricity price data starts in 2007. 
### Therefore, the processed CO₂ dataset retains the full historical period for standalone trend analysis. 
### The integrated analytical dataset uses the common period (2007–2024) to ensure valid comparisons between 
### emissions and electricity prices.

## Merging datasets

The processed datasets are merged using an inner join on `country` and `year`.

An inner join is used because comparable electricity price data is available only from 2007 onwards. The merged dataset therefore contains the common period (2007–2024), while the complete CO₂ dataset (1990–2024) remains available for long-term trend analysis.

In [11]:
df_co2_electricity_merged = df_electricity.merge(
    df_co2,
    on=["country","year"],
    how="inner"
)

### Checking quality of merged DataFrame

In [12]:
df_co2_electricity_merged.shape

(36, 6)

In [13]:
df_co2_electricity_merged.head()

,country,year,electricity_price,population,co2,co2_per_capita
0,European Union (27),2007,0.1682,436994510.0,3701.867,8.471
1,European Union (27),2008,0.1640,438548227.0,3625.826,8.268
2,European Union (27),2009,0.1674,439690812.0,3330.504,7.575
3,European Union (27),2010,0.1751,440608965.0,3424.327,7.772
4,European Union (27),2011,0.1872,441440625.0,3327.059,7.537


In [14]:
df_co2_electricity_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   country            36 non-null     object 
 1   year               36 non-null     int64  
 2   electricity_price  36 non-null     float64
 3   population         36 non-null     float64
 4   co2                36 non-null     float64
 5   co2_per_capita     36 non-null     float64
dtypes: float64(4), int64(1), object(1)
memory usage: 1.8+ KB


In [15]:
df_co2_electricity_merged.isna().sum()

country              0
year                 0
electricity_price    0
population           0
co2                  0
co2_per_capita       0
dtype: int64

## Saving data

In [19]:
df_co2_electricity_merged.to_csv("../data/processed/energy_transition_merged.csv",
index = False
)